In [ ]:
import os
import pandas as pd
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Upload Widget
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

# Function to handle uploaded file and generate JSON
def process_uploaded_file(change):
    uploaded_file = next(iter(upload_button.value))
    file_name = uploaded_file
    file_content = upload_button.value[file_name]['content']
    file_path = f"/content/{file_name}"
    with open(file_path, "wb") as f:
        f.write(file_content)

    print(f"File uploaded successfully: {file_path}")

    try:
        df = pd.read_excel(file_path)
        levels = [f"Level {i}" for i in range(1, 11)]
        required_columns = levels + ["ID. Figura", "Description", "Fabricante/Fornecedor",
                                     "Referência comercial", "Código SAP", "Qtd", "Documentação",
                                     "Image URL1", "Image URL2", "Image URL3"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {missing_columns}")
            return
        df = df.astype(str).replace("nan", "").fillna("")

        def transform_url(url):
            if "https://drive.google.com/file/d/" in url and "/view?usp=drive_link" in url:
                file_id = url.split("/file/d/")[1].split("/view?usp=drive_link")[0]
                return f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
            return url

        def calculate_cadastro(subset):
            total = len(subset)
            non_blank = subset["Código SAP"].apply(lambda x: x != "").sum()
            percentage = (non_blank / total) * 100 if total > 0 else 0
            return f"{non_blank}/{total} ({percentage:.1f}%)"

        def build_hierarchy(df, levels, parent_level=0):
            if parent_level >= len(levels):
                return []
            hierarchy = []
            unique_items = df[levels[parent_level]].unique()
            for item in unique_items:
                if not item:
                    continue
                subset = df[df[levels[parent_level]] == item]
                id_figura = subset["ID. Figura"].iloc[0]
                id_figura = int(float(id_figura)) if id_figura.replace('.', '', 1).isdigit() else None
                node_name = f"{item} [{id_figura}]" if id_figura else item

                # Convert 'codigoSap' and 'qtd' to integers (or 0 if invalid)
                def to_int(value):
                    try:
                        return int(float(value.replace(',', '').strip()))
                    except ValueError:
                        return 0

                node = {
                    "name": node_name,
                    "originalname": item,
                    "idFigura": id_figura,
                    "description": subset["Description"].iloc[0],
                    "fabricanteFornecedor": subset["Fabricante/Fornecedor"].iloc[0],
                    "referenciaComercial": subset["Referência comercial"].iloc[0],
                    "codigoSap": to_int(subset["Código SAP"].iloc[0]),
                    "qtd": to_int(subset["Qtd"].iloc[0]),
                    "documentation": subset["Documentação"].iloc[0],
                    "imageUrl1": transform_url(subset["Image URL1"].iloc[0]),
                    "imageUrl2": transform_url(subset["Image URL2"].iloc[0]),
                    "imageUrl3": transform_url(subset["Image URL3"].iloc[0]),
                    "cadastro": calculate_cadastro(subset),
                    "children": build_hierarchy(subset, levels, parent_level + 1)
                }
                hierarchy.append(node)
            return hierarchy

        hierarchy = build_hierarchy(df, levels)
        output_json_path = f"/content/{file_name.replace('.xlsx', '.json')}"
        with open(output_json_path, "w") as f:
            json.dump(hierarchy, f, indent=4)

        print(f"Hierarchy JSON saved at: {output_json_path}")
        files.download(output_json_path)

    except Exception as e:
        print(f"Error processing file: {e}")

upload_button.observe(process_uploaded_file, names='value')


FileUpload(value={}, accept='.xlsx', description='Upload')

In [ ]:
# v.2 - Transforma em Maiúsculo

import os
import pandas as pd
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Upload Widget
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

# Function to handle uploaded file and generate JSON
def process_uploaded_file(change):
    uploaded_file = next(iter(upload_button.value))
    file_name = uploaded_file
    file_content = upload_button.value[file_name]['content']
    file_path = f"/content/{file_name}"
    with open(file_path, "wb") as f:
        f.write(file_content)

    print(f"File uploaded successfully: {file_path}")

    try:
        df = pd.read_excel(file_path)
        levels = [f"Level {i}" for i in range(1, 11)]
        required_columns = levels + ["ID. Figura", "Description", "Fabricante/Fornecedor",
                                     "Referência comercial", "Código SAP", "Qtd", "Documentação",
                                     "Image URL1", "Image URL2", "Image URL3"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {missing_columns}")
            return

        # Convert all columns to string and handle NaN values
        df = df.astype(str).replace("nan", "").fillna("")

        # Convert level column values to uppercase using map()
        for level in levels:
            df[level] = df[level].map(lambda x: x.upper() if isinstance(x, str) else x)

        def transform_url(url):
            if "https://drive.google.com/file/d/" in url and "/view?usp=drive_link" in url:
                file_id = url.split("/file/d/")[1].split("/view?usp=drive_link")[0]
                return f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
            return url

        def calculate_cadastro(subset):
            total = len(subset)
            non_blank = subset["Código SAP"].apply(lambda x: x != "").sum()
            percentage = (non_blank / total) * 100 if total > 0 else 0
            return f"{non_blank}/{total} ({percentage:.1f}%)"

        def build_hierarchy(df, levels, parent_level=0):
            if parent_level >= len(levels):
                return []
            hierarchy = []
            unique_items = df[levels[parent_level]].unique()
            for item in unique_items:
                if not item:
                    continue
                subset = df[df[levels[parent_level]] == item]
                id_figura = subset["ID. Figura"].iloc[0]
                id_figura = int(float(id_figura)) if id_figura.replace('.', '', 1).isdigit() else None
                node_name = f"{item} [{id_figura}]" if id_figura else item

                # Convert 'codigoSap' and 'qtd' to integers (or 0 if invalid)
                def to_int(value):
                    try:
                        return int(float(value.replace(',', '').strip()))
                    except ValueError:
                        return 0

                node = {
                    "name": node_name.upper(),  # Convert to uppercase
                    "originalname": item.upper(),  # Convert to uppercase
                    "idFigura": id_figura,
                    "description": subset["Description"].iloc[0],
                    "fabricanteFornecedor": subset["Fabricante/Fornecedor"].iloc[0],
                    "referenciaComercial": subset["Referência comercial"].iloc[0],
                    "codigoSap": to_int(subset["Código SAP"].iloc[0]),
                    "qtd": to_int(subset["Qtd"].iloc[0]),
                    "documentation": subset["Documentação"].iloc[0],
                    "imageUrl1": transform_url(subset["Image URL1"].iloc[0]),
                    "imageUrl2": transform_url(subset["Image URL2"].iloc[0]),
                    "imageUrl3": transform_url(subset["Image URL3"].iloc[0]),
                    "cadastro": calculate_cadastro(subset),
                    "children": build_hierarchy(subset, levels, parent_level + 1)
                }
                hierarchy.append(node)
            return hierarchy

        hierarchy = build_hierarchy(df, levels)
        output_json_path = f"/content/{file_name.replace('.xlsx', '.json')}"
        with open(output_json_path, "w") as f:
            json.dump(hierarchy, f, indent=4)

        print(f"Hierarchy JSON saved at: {output_json_path}")
        files.download(output_json_path)

    except Exception as e:
        print(f"Error processing file: {e}")

upload_button.observe(process_uploaded_file, names='value')


FileUpload(value={}, accept='.xlsx', description='Upload')

File uploaded successfully: /content/1MRO_FROTA.7000 (2).xlsx
Hierarchy JSON saved at: /content/1MRO_FROTA.7000 (2).json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import pandas as pd
import json
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Upload Widget
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

# Function to handle uploaded file and generate JSON
def process_uploaded_file(change):
    uploaded_file = next(iter(upload_button.value))
    file_name = uploaded_file
    file_content = upload_button.value[file_name]['content']
    file_path = f"/content/{file_name}"
    with open(file_path, "wb") as f:
        f.write(file_content)

    print(f"File uploaded successfully: {file_path}")

    try:
        df = pd.read_excel(file_path)
        levels = [f"Level {i}" for i in range(1, 11)]
        required_columns = levels + ["ID. Figura", "Description", "Fabricante/Fornecedor",
                                     "Referência comercial", "Código SAP", "Qtd", "Documentação",
                                     "Image URL1", "Image URL2", "Image URL3"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {missing_columns}")
            return
        df = df.astype(str).replace("nan", "").fillna("")

        def transform_url(url):
            if "https://drive.google.com/file/d/" in url and "/view?usp=drive_link" in url:
                file_id = url.split("/file/d/")[1].split("/view?usp=drive_link")[0]
                return f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
            return url

        def calculate_cadastro(subset):
            total = len(subset)
            non_blank = subset["Código SAP"].apply(lambda x: x != "").sum()
            percentage = (non_blank / total) * 100 if total > 0 else 0
            return f"{non_blank}/{total} ({percentage:.1f}%)"

        def build_hierarchy(df, levels, parent_level=0):
            if parent_level >= len(levels):
                return []
            hierarchy = []
            unique_items = df[levels[parent_level]].unique()
            for item in unique_items:
                if not item:
                    continue
                subset = df[df[levels[parent_level]] == item]
                id_figura = subset["ID. Figura"].iloc[0]
                id_figura = int(float(id_figura)) if id_figura.replace('.', '', 1).isdigit() else None
                node_name = f"{item} [{id_figura}]" if id_figura else item

                # Convert 'codigoSap' and 'qtd' to integers (or 0 if invalid)
                def to_int(value):
                    try:
                        return int(float(value.replace(',', '').strip()))
                    except ValueError:
                        return 0

                node = {
                    "name": node_name,
                    "originalname": item,
                    "idFigura": id_figura,
                    "description": subset["Description"].iloc[0],
                    "fabricanteFornecedor": subset["Fabricante/Fornecedor"].iloc[0],
                    "referenciaComercial": subset["Referência comercial"].iloc[0],
                    "codigoSap": to_int(subset["Código SAP"].iloc[0]),
                    "qtd": to_int(subset["Qtd"].iloc[0]),
                    "documentation": subset["Documentação"].iloc[0],
                    "imageUrl1": transform_url(subset["Image URL1"].iloc[0]),
                    "imageUrl2": transform_url(subset["Image URL2"].iloc[0]),
                    "imageUrl3": transform_url(subset["Image URL3"].iloc[0]),
                    "cadastro": calculate_cadastro(subset),
                    "children": build_hierarchy(subset, levels, parent_level + 1)
                }
                hierarchy.append(node)
            return hierarchy

        hierarchy = build_hierarchy(df, levels)
        output_json_path = f"/content/{file_name.replace('.xlsx', '.json')}"
        with open(output_json_path, "w") as f:
            json.dump(hierarchy, f, indent=4)

        print(f"Hierarchy JSON saved at: {output_json_path}")
        files.download(output_json_path)

    except Exception as e:
        print(f"Error processing file: {e}")

upload_button.observe(process_uploaded_file, names='value')

FileUpload(value={}, accept='.xlsx', description='Upload')

File uploaded successfully: /content/1MRO_FROTA.7000 (2).xlsx
Hierarchy JSON saved at: /content/1MRO_FROTA.7000 (2).json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# REMOVE CARACTERES ESPECIAIS - VERSÃO PRODUÇÃO (usar esta)

import os
import pandas as pd
import json
import ipywidgets as widgets
import unicodedata
import re
from IPython.display import display
from google.colab import files

# Upload Widget
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

# Function to normalize text (remove special characters)
def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')  # Remove accents
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove any remaining special characters
    return text.upper()  # Convert to uppercase

# Function to handle uploaded file and generate JSON
def process_uploaded_file(change):
    uploaded_file = next(iter(upload_button.value))
    file_name = uploaded_file
    file_content = upload_button.value[file_name]['content']
    file_path = f"/content/{file_name}"
    with open(file_path, "wb") as f:
        f.write(file_content)

    print(f"File uploaded successfully: {file_path}")

    try:
        df = pd.read_excel(file_path)
        levels = [f"Level {i}" for i in range(1, 11)]
        required_columns = levels + ["ID. Figura", "Description", "Fabricante/Fornecedor",
                                     "Referência comercial", "Código SAP", "Qtd", "Documentação",
                                     "Image URL1", "Image URL2", "Image URL3"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {missing_columns}")
            return

        # Convert all columns to string and handle NaN values
        df = df.astype(str).replace("nan", "").fillna("")

        # Normalize values in the levels columns
        for level in levels:
            df[level] = df[level].map(lambda x: normalize_text(x) if isinstance(x, str) else x)

        # Normalize description column
        df["Description"] = df["Description"].map(lambda x: normalize_text(x) if isinstance(x, str) else x)

        def transform_url(url):
            if "https://drive.google.com/file/d/" in url and "/view?usp=drive_link" in url:
                file_id = url.split("/file/d/")[1].split("/view?usp=drive_link")[0]
                return f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
            return url

        def calculate_cadastro(subset):
            total = len(subset)
            non_blank = subset["Código SAP"].apply(lambda x: x != "").sum()
            percentage = (non_blank / total) * 100 if total > 0 else 0
            return f"{non_blank}/{total} ({percentage:.1f}%)"

        def build_hierarchy(df, levels, parent_level=0):
            if parent_level >= len(levels):
                return []
            hierarchy = []
            unique_items = df[levels[parent_level]].unique()
            for item in unique_items:
                if not item:
                    continue
                subset = df[df[levels[parent_level]] == item]
                id_figura = subset["ID. Figura"].iloc[0]
                id_figura = int(float(id_figura)) if id_figura.replace('.', '', 1).isdigit() else None
                node_name = f"{item} [{id_figura}]" if id_figura else item

                # Convert 'codigoSap' and 'qtd' to integers (or 0 if invalid)
                def to_int(value):
                    try:
                        return int(float(value.replace(',', '').strip()))
                    except ValueError:
                        return 0

                node = {
                    "name": normalize_text(node_name),  # Normalize and convert to uppercase
                    "originalname": normalize_text(item),  # Normalize and convert to uppercase
                    "idFigura": id_figura,
                    "description": normalize_text(subset["Description"].iloc[0]),  # Normalize description
                    "fabricanteFornecedor": subset["Fabricante/Fornecedor"].iloc[0],
                    "referenciaComercial": subset["Referência comercial"].iloc[0],
                    "codigoSap": to_int(subset["Código SAP"].iloc[0]),
                    "qtd": to_int(subset["Qtd"].iloc[0]),
                    "documentation": subset["Documentação"].iloc[0],
                    "imageUrl1": transform_url(subset["Image URL1"].iloc[0]),
                    "imageUrl2": transform_url(subset["Image URL2"].iloc[0]),
                    "imageUrl3": transform_url(subset["Image URL3"].iloc[0]),
                    "cadastro": calculate_cadastro(subset),
                    "children": build_hierarchy(subset, levels, parent_level + 1)
                }
                hierarchy.append(node)
            return hierarchy

        hierarchy = build_hierarchy(df, levels)
        output_json_path = f"/content/{file_name.replace('.xlsx', '.json')}"
        with open(output_json_path, "w") as f:
            json.dump(hierarchy, f, indent=4)

        print(f"Hierarchy JSON saved at: {output_json_path}")
        files.download(output_json_path)

    except Exception as e:
        print(f"Error processing file: {e}")

upload_button.observe(process_uploaded_file, names='value')


FileUpload(value={}, accept='.xlsx', description='Upload')

File uploaded successfully: /content/1MRO_FROTA.7000 (2).xlsx
Hierarchy JSON saved at: /content/1MRO_FROTA.7000 (2).json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# VERSÃO MAIO.2025 - USAR ESTA

import os
import pandas as pd
import json
import ipywidgets as widgets
import unicodedata
import re
from IPython.display import display
from google.colab import files

# Upload Widget
upload_button = widgets.FileUpload(accept='.xlsx', multiple=False)
display(upload_button)

# Função de normalização de texto
def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')  # Remove acentos
    text = re.sub(r'[^a-zA-Z0-9\s\-]', '', text)  # Remove caracteres especiais
    return text.upper()  # Converte para maiúsculo

# Função principal de processamento do arquivo
def process_uploaded_file(change):
    uploaded_file = next(iter(upload_button.value))
    file_name = uploaded_file
    file_content = upload_button.value[file_name]['content']
    file_path = f"/content/{file_name}"
    with open(file_path, "wb") as f:
        f.write(file_content)

    print(f"File uploaded successfully: {file_path}")

    try:
        df = pd.read_excel(file_path)
        levels = [f"Level {i}" for i in range(1, 11)]
        required_columns = levels + ["ID. Figura", "Description", "Fabricante/Fornecedor",
                                     "Referência comercial", "Código SAP", "Qtd", "Documentação",
                                     "Image URL1", "Image URL2", "Image URL3"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing required columns: {missing_columns}")
            return

        # Tratamento de strings e valores nulos
        df = df.astype(str).replace("nan", "").fillna("")

        def transform_url(url):
            if "https://drive.google.com/file/d/" in url and "/view?usp=drive_link" in url:
                file_id = url.split("/file/d/")[1].split("/view?usp=drive_link")[0]
                return f"https://drive.google.com/thumbnail?id={file_id}&sz=w800"
            return url

        def calculate_cadastro(subset):
            total = len(subset)
            non_blank = subset["Código SAP"].apply(lambda x: x != "").sum()
            percentage = (non_blank / total) * 100 if total > 0 else 0
            return f"{non_blank}/{total} ({percentage:.1f}%)"

        def build_hierarchy(df, levels, parent_level=0):
            if parent_level >= len(levels):
                return []
            hierarchy = []
            unique_items = df[levels[parent_level]].unique()
            for item in unique_items:
                if not item:
                    continue
                subset = df[df[levels[parent_level]] == item]
                id_figura = subset["ID. Figura"].iloc[0]
                id_figura = int(float(id_figura)) if id_figura.replace('.', '', 1).isdigit() else None
                node_name = f"{item} [{id_figura}]" if id_figura else item

                def to_int(value):
                    try:
                        return int(float(value.replace(',', '').strip()))
                    except ValueError:
                        return 0

                node = {
                    "name": normalize_text(node_name),
                    "originalname": normalize_text(item),
                    "idFigura": id_figura,
                    "description": normalize_text(subset["Description"].iloc[0]),
                    "fabricanteFornecedor": subset["Fabricante/Fornecedor"].iloc[0],
                    "referenciaComercial": subset["Referência comercial"].iloc[0],
                    "codigoSap": to_int(subset["Código SAP"].iloc[0]),
                    "qtd": to_int(subset["Qtd"].iloc[0]),
                    "documentation": subset["Documentação"].iloc[0],
                    "imageUrl1": transform_url(subset["Image URL1"].iloc[0]),
                    "imageUrl2": transform_url(subset["Image URL2"].iloc[0]),
                    "imageUrl3": transform_url(subset["Image URL3"].iloc[0]),
                    "cadastro": calculate_cadastro(subset),
                    "children": build_hierarchy(subset, levels, parent_level + 1)
                }
                hierarchy.append(node)
            return hierarchy

        hierarchy = build_hierarchy(df, levels)
        output_json_path = f"/content/{file_name.replace('.xlsx', '.json')}"
        with open(output_json_path, "w") as f:
            json.dump(hierarchy, f, indent=4)

        print(f"Hierarchy JSON saved at: {output_json_path}")
        files.download(output_json_path)

    except Exception as e:
        print(f"Error processing file: {e}")

# Observador para disparar a função ao upload
upload_button.observe(process_uploaded_file, names='value')


FileUpload(value={}, accept='.xlsx', description='Upload')

File uploaded successfully: /content/SISTEMA_EM BRANCO.xlsx
Hierarchy JSON saved at: /content/SISTEMA_EM BRANCO.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>